In [ ]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

dry_bean = fetch_ucirepo(id=602)
df = pd.concat([dry_bean.data.features, dry_bean.data.targets], axis=1)

df = df.rename(columns={"AspectRation": "AspectRatio", "roundness": "Roundness"})

TARGET = "Class"
FEATURES = ["Area", "Perimeter", "MajorAxisLength", "MinorAxisLength",
    "AspectRatio", "Eccentricity", "ConvexArea", "EquivDiameter", "Extent", "Solidity",
    "Roundness", "Compactness", "ShapeFactor1", "ShapeFactor2", "ShapeFactor3", "ShapeFactor4"]

### Reconhecimento inicial

In [9]:

print(f'O Data frame escolhido contem uma série de dados sobre um conjunto extenso de feijões, cada coluna do data frame representa a medida de uma caracteristica, sendo a ultima coluna representando a classe de cada feijão')
print(f'\n  Distribuidas entre {len(df[TARGET].unique())} classes, para cada uma das {df.shape[0]} amostras foram realizadas medições para {df.shape[1] -1 } caracteristicas')


O Data frame escolhido contem uma série de dados sobre um conjunto extenso de feijões, cada coluna do data frame representa a medida de uma caracteristica, sendo a ultima coluna representando a classe de cada feijão

  Distribuidas entre 7 classes, para cada uma das 13611 amostras foram realizadas medições para 16 caracteristicas


In [19]:
import sys
from pathlib import Path


raiz_projeto = Path('.').resolve().parent.parent

if str(raiz_projeto) not in sys.path:
    sys.path.append(str(raiz_projeto))

### Usando a biblioteca para particionar os dados:

In [26]:
from rbg.Utilities import train_test_split


X = df.iloc[:,:-1]
Y = df[TARGET]

x_train,x_test,y_train,y_test =train_test_split(X,Y)

### Construindo a rede:

In [38]:
from rbg.NeuralNetwork import MLPClassifier
import numpy as np

Rede = MLPClassifier(
    hidden_layers=(X.shape[1],int(X.shape[1]*1.5),X.shape[1],len(df[TARGET].unique())),
    activation='relu',
    optimizer="adam",
    learning_rate=0.001,
    epochs=10000,
    batch_size=32
)


In [39]:

Rede.fit(x_train,y_train)

y_predicter = Rede.predict(x_test)

In [35]:
classes = dict()
for indx,t in enumerate(df[TARGET].unique()):
    classes[t] = indx

size_cl = len(classes)

In [36]:
matriz_confusão = np.zeros((size_cl,size_cl))


for train,test in zip(y_predicter,y_test):
    matriz_confusão[classes[train],classes[test]] += 1

print(matriz_confusão)

[[  0.   0.   0.   0.   0.   0.   0.]
 [  0.   0.   0.   0.   0.   0.   0.]
 [  0.   0.   0.   0.   0.   0.   0.]
 [  0.   0.   0.   0.   0.   0.   0.]
 [  0.   0.   0.   0.   0.   0.   0.]
 [  0.   0.   0.   0.   0.   0.   0.]
 [404. 267. 104. 350. 378. 508. 712.]]


In [37]:
# 1. Inverte o dicionário para buscar o nome da classe pelo índice
idx_para_classe = {idx: nome for nome, idx in classes.items()}

print("\n" + "="*40)
print("RELATÓRIO DE CLASSIFICAÇÃO")
print("="*40)

# Acertos globais são a soma da diagonal principal
acertos_totais = np.trace(matriz_confusão)
total_amostras = np.sum(matriz_confusão)
print(f"Acurácia Geral: {int(acertos_totais)}/{int(total_amostras)} ({(acertos_totais/total_amostras):.2%})\n")

# 2. Itera sobre cada classe para extrair as métricas da matriz
for i in range(size_cl):
    nome_classe = idx_para_classe[i]
    
    # Acertos estão na diagonal [i, i]
    acertos = matriz_confusão[i, i]
    
    # Total real da classe é a soma da COLUNA i
    total_real = np.sum(matriz_confusão[:, i]) 
    
    # Total que a rede previu como sendo dessa classe é a soma da LINHA i
    total_previsto = np.sum(matriz_confusão[i, :])
    
    taxa_acerto = (acertos / total_real) if total_real > 0 else 0
    
    print(f"Classe: '{nome_classe}'")
    print(f"  -> Acertos: {int(acertos)} de {int(total_real)} exemplos reais ({taxa_acerto:.2%})")
    
    # Detalhando os erros
    falsos_negativos = int(total_real - acertos)
    falsos_positivos = int(total_previsto - acertos)
    
    if falsos_negativos > 0:
        print(f"  -> A rede errou {falsos_negativos} exemplos que eram '{nome_classe}' (classificou como outra coisa).")
    
    if falsos_positivos > 0:
        print(f"  -> A rede errou ao classificar {falsos_positivos} exemplos de outras classes como se fossem '{nome_classe}'.")
        
    print("-" * 40)


RELATÓRIO DE CLASSIFICAÇÃO
Acurácia Geral: 712/2723 (26.15%)

Classe: 'SEKER'
  -> Acertos: 0 de 404 exemplos reais (0.00%)
  -> A rede errou 404 exemplos que eram 'SEKER' (classificou como outra coisa).
----------------------------------------
Classe: 'BARBUNYA'
  -> Acertos: 0 de 267 exemplos reais (0.00%)
  -> A rede errou 267 exemplos que eram 'BARBUNYA' (classificou como outra coisa).
----------------------------------------
Classe: 'BOMBAY'
  -> Acertos: 0 de 104 exemplos reais (0.00%)
  -> A rede errou 104 exemplos que eram 'BOMBAY' (classificou como outra coisa).
----------------------------------------
Classe: 'CALI'
  -> Acertos: 0 de 350 exemplos reais (0.00%)
  -> A rede errou 350 exemplos que eram 'CALI' (classificou como outra coisa).
----------------------------------------
Classe: 'HOROZ'
  -> Acertos: 0 de 378 exemplos reais (0.00%)
  -> A rede errou 378 exemplos que eram 'HOROZ' (classificou como outra coisa).
----------------------------------------
Classe: 'SIRA'
 